# Interactive formation recovery: two-impulse vs Chernick–D'Amico vs SCP

Same scenario as `examples/sweep.py`: a 3-satellite along-track formation, 1 km
apart, 550 km equatorial orbit. Sat-1 takes an unplanned impulse at t = 0, drifts
for the **reaction delay**, then returns to its slot within the **transfer time**.
Both sliders are in orbits. Three controllers fly the same initial conditions:

- **Two-impulse CW**: closed-form transfer, burns fixed at the start and end of the window.
- **Chernick–D'Amico ROE**: closed-form fuel-optimal scheme in relative orbital elements
  (`cwsim.roe`), tangential in-plane burns plus one cross-track burn at optimal times.
- **SCP**: sequential convex programming over the whole formation (`cwsim.scp`), with the
  keep-out sphere enforced. By default every satellite keeps its own slot, so the problem
  matches the two baselines; tick *SCP re-slotting* to let satellites swap slots.

The first figure compares trajectories. The second compares fuel use: each burn's size
against the per-burn limit, and the cumulative Δv over time. A summary table prints below.
SCP takes a few seconds per setting; untick *run SCP* for fast scrubbing.

In [4]:
import sys
from functools import lru_cache
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle

ROOT = Path.cwd().resolve()
if not (ROOT / "cwsim").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "examples"))

from cwsim import (
    CircularOrbit, ClosedFormUnavailable, along_track_formation, cw_stm,
    cw_two_impulse, propagate_cw_with_impulses, roe_impulsive_reconfig,
)
from cwsim.scp import ScpConfig, plan_recovery
from sweep import (
    BASELINE_AXIS, CHECK_DT, D_MIN, DIRECTIONS, DRIFT, DV_MAX, GRID, INK, INK2,
    MUTED, SPACING, SURFACE, TRANSFER, cw_positions, min_separation, style_axis,
)

chief = CircularOrbit(altitude=550e3, inclination=0.0)
n, period = chief.mean_motion, chief.period
sats = along_track_formation(spacing=SPACING, count=3)
target = sats[0].x0
others = {s.name: s.x0[:3] for s in sats[1:]}

# One fixed style per controller, reused in both figures. Colors are slots
# 1, 3 and 7 of the reference categorical palette (orange, slot 2, is the
# shared drift leg); each also gets its own dash and marker so identity
# never rests on color alone.
METHODS = {
    "two_impulse": dict(label="Two-impulse CW", short="2-impulse", color=TRANSFER,
                        ls="-", marker="o"),
    "roe": dict(label="Chernick–D'Amico ROE", short="ROE", color="#1baf7a",
                ls=(0, (6, 2)), marker="D"),
    "scp": dict(label="SCP", short="SCP", color="#6250d6",
                ls=(0, (1.5, 1.5)), marker="^"),
}

# Coarser than the ScpConfig defaults so a slider move solves in seconds.
SCP_CFG = ScpConfig(dt=120.0, n_sub=4, collision_margin=10.0, max_iters=6,
                    d_min=D_MIN, dv_max=DV_MAX, check_dt=CHECK_DT)

In [5]:
SINGULAR_DV = 100 * DV_MAX  # [m/s]


def _leg(x_delay, t_delay, horizon, burns, drift_sep):
    """Sat-1 transfer leg for a burn list [(t_abs, dv)] starting at t_delay."""
    rel = [(t - t_delay, dv) for t, dv in burns]
    tau = np.linspace(0.0, horizon, int(np.ceil(horizon / CHECK_DT)) + 1)
    r = propagate_cw_with_impulses(x_delay, n, tau, rel)[:, :3]
    burn_pos = propagate_cw_with_impulses(
        x_delay, n, np.array([t for t, _ in rel]), rel)[:, :3]
    mags = [float(np.linalg.norm(dv)) for _, dv in burns]
    return dict(r_transfer=r, burn_times=[t for t, _ in burns],
                burn_mags=mags, burn_pos=burn_pos, total=sum(mags),
                sep=min(drift_sep, min_separation(r, others.values())),
                moved={}, note="")


@lru_cache(maxsize=64)
def simulate(direction, mag, delay_orb, transfer_orb, run_scp=True,
             reslot=False):
    """Drift, then all three recoveries from the same state at t_delay.

    Returns dict(r_drift, x_delay, t_delay, tf, methods={key: result}); a
    method that cannot solve this case holds dict(error=...) instead.
    """
    hit = mag * DIRECTIONS[direction]
    x_hit = target.copy()
    x_hit[3:] += hit
    t_delay = delay_orb * period
    horizon = transfer_orb * period
    tf = t_delay + horizon

    t_drift = np.linspace(0.0, t_delay,
                          max(int(np.ceil(t_delay / CHECK_DT)) + 1, 2))
    r_drift = cw_positions(x_hit, n, t_drift)
    drift_sep = min_separation(r_drift, others.values())
    x_delay = cw_stm(n, t_delay) @ x_hit
    out = {}

    # Two-impulse CW. Near whole half-orbit transfer times the solve is
    # ill-conditioned rather than exactly singular: an absurd delta-v.
    try:
        dv1, dv2 = cw_two_impulse(x_delay, n, horizon, target)
        if np.linalg.norm(dv1) + np.linalg.norm(dv2) > SINGULAR_DV:
            raise np.linalg.LinAlgError
        out["two_impulse"] = _leg(x_delay, t_delay, horizon,
                                  [(t_delay, dv1), (tf, dv2)], drift_sep)
    except np.linalg.LinAlgError:
        out["two_impulse"] = dict(error="singular transfer time")

    # Chernick & D'Amico closed form. u0 is the chief argument of latitude
    # at the first possible burn (it only rotates the ROE vectors).
    try:
        plan = roe_impulsive_reconfig(x_delay, n, horizon, target,
                                      u0=chief.arg_lat0 + n * t_delay)
        res = _leg(x_delay, t_delay, horizon,
                   [(t_delay + t, dv) for t, dv in plan.burns], drift_sep)
        if plan.dominant == "none":
            res["note"] = "out-of-plane only, optimal"
        else:
            res["note"] = (f"{plan.dominant}-dominant, "
                           + ("optimal" if plan.inplane_optimal else
                              f"in-plane ×{plan.inplane_ratio:.2f} of dv_min"))
        out["roe"] = res
    except ClosedFormUnavailable:
        out["roe"] = dict(error="window too short for closed form")

    # SCP over the whole formation (other satellites may burn too).
    if run_scp:
        fleet = along_track_formation(spacing=SPACING, count=3)
        fleet[0].add_impulse(0.0, hit)
        try:
            plan = plan_recovery(chief, fleet, t_delay, tf, SCP_CFG,
                                 n_assignments=2, allow_reslot=reslot)
        except RuntimeError:
            out["scp"] = dict(error="SCP failed")
        else:
            mask = plan.times >= t_delay - 1e-9
            burns = [(t, dv, name) for name, bl in plan.burns.items()
                     for t, dv in bl]
            pos = []
            for t, _, name in burns:  # positions at the burn instants
                sat = next(s for s in fleet if s.name == name)
                pos.append(propagate_cw_with_impulses(
                    sat.x0, n, np.array([t]),
                    list(sat.impulses) + plan.burns[name])[0, :3])
            moved = {name: tr[mask, :3]
                     for name, tr in plan.trajectories.items()
                     if name != "Sat-1"
                     and np.abs(tr[:, :3] - tr[0, :3]).max() > 1.0}
            out["scp"] = dict(
                r_transfer=plan.trajectories["Sat-1"][mask, :3],
                burn_times=[t for t, _, _ in burns],
                burn_mags=[float(np.linalg.norm(dv)) for _, dv, _ in burns],
                burn_pos=np.array(pos).reshape(-1, 3),
                total=plan.total_dv, sep=plan.min_separation, moved=moved,
                note=(f"slots {plan.assignment}, "
                      + ("converged" if plan.converged else "not converged")
                      + (f", slack {plan.slack_max:.1f} m"
                         if plan.slack_max > 1e-3 else "")),
            )
    return dict(r_drift=r_drift, x_delay=x_delay, t_delay=t_delay, tf=tf,
                methods=out)


def _flags(res):
    flags = []
    if res["burn_mags"] and max(res["burn_mags"]) > DV_MAX:
        flags.append(f"burn > {DV_MAX:g} m/s")
    if res["sep"] < D_MIN:
        flags.append(f"keep-out < {D_MIN:g} m")
    return ", ".join(flags) or "feasible"


def plot_trajectories(sim, direction, mag, delay_orb, transfer_orb):
    fig, axes = plt.subplots(2, 1, figsize=(11, 7.5), facecolor=SURFACE,
                             constrained_layout=True, sharex=True)
    # Top: radial x vs along-track y. Bottom: cross-track z vs along-track y.
    for ax, k, label in ((axes[0], 0, "radial x [m]"),
                         (axes[1], 2, "cross-track z [m]")):
        style_axis(ax)
        ax.grid(True, color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        for name, p in others.items():
            ax.add_patch(Circle((p[1], p[k]), D_MIN, facecolor=GRID,
                                edgecolor=MUTED, linestyle=(0, (4, 3)),
                                linewidth=1.0, zorder=1))
            ax.plot(p[1], p[k], "s", color=INK2, markersize=7, zorder=3)
            ax.annotate(name, (p[1], p[k]), xytext=(0, -16),
                        textcoords="offset points", ha="center", fontsize=8,
                        color=INK2)
        ax.plot(sim["r_drift"][:, 1], sim["r_drift"][:, k], color=DRIFT,
                linewidth=2, linestyle=(0, (4, 2)), zorder=2,
                label=f"drift, {delay_orb:.2f} orbit")
        for key, st in METHODS.items():
            res = sim["methods"].get(key)
            if res is None or "error" in res:
                continue
            ax.plot(res["r_transfer"][:, 1], res["r_transfer"][:, k],
                    color=st["color"], linestyle=st["ls"], linewidth=2,
                    zorder=2, label=f"{st['label']}, {res['total']:.3f} m/s")
            for name, r in res["moved"].items():  # SCP neighbours that burned
                ax.plot(r[:, 1], r[:, k], color=st["color"], linestyle=st["ls"],
                        linewidth=1.2, alpha=0.7, zorder=2)
            ax.plot(res["burn_pos"][:, 1], res["burn_pos"][:, k], st["marker"],
                    color=st["color"], markersize=8, markeredgecolor=SURFACE,
                    markeredgewidth=1.5, linestyle="none", zorder=5)
        ax.plot(target[1], target[k], "o", markersize=9,
                markerfacecolor=SURFACE, markeredgecolor=INK,
                markeredgewidth=1.6, zorder=4)
        ax.plot(sim["x_delay"][1], sim["x_delay"][k], "o", color=DRIFT,
                markersize=8, markeredgecolor=SURFACE, markeredgewidth=1.2,
                zorder=4)
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_ylabel(label, fontsize=9, color=INK2)
    axes[0].invert_xaxis()  # +y to the left (velocity direction left)
    axes[1].set_xlabel("along-track y [m]  (+y ←)", fontsize=9, color=INK2)
    axes[0].legend(frameon=False, fontsize=8, labelcolor=INK2,
                   loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[0].set_title(
        f"Recovery trajectories, {direction} hit (|δv| = {mag:g} m/s), "
        f"transfer {transfer_orb:.2f} orbit · markers are burns",
        fontsize=10, color=INK, loc="left")
    plt.show()


def plot_fuel(sim, delay_orb, transfer_orb):
    t_end = sim["tf"] / period
    fig, (ax_b, ax_c) = plt.subplots(1, 2, figsize=(11, 4.1), sharex=True,
                                     facecolor=SURFACE, constrained_layout=True)
    for ax in (ax_b, ax_c):
        style_axis(ax)
        ax.grid(True, axis="y", color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        ax.axvspan(0.0, delay_orb, color=GRID, alpha=0.6, linewidth=0, zorder=0)
        ax.set_xlim(0.0, t_end * 1.02)
        ax.set_xlabel("time since hit [orbits]", fontsize=9, color=INK2)
    if delay_orb > 0:
        ax_b.text(delay_orb / 2, 0.03, "drift\n(no burns)",
                  transform=ax_b.get_xaxis_transform(), ha="center",
                  va="bottom", fontsize=8, color=MUTED)
    ax_b.axhline(DV_MAX, color=MUTED, linestyle=(0, (4, 3)), linewidth=1.0)
    ax_b.annotate(f"per-burn limit {DV_MAX:g} m/s", (t_end * 1.02, DV_MAX),
                  xytext=(-4, 4), textcoords="offset points", ha="right",
                  fontsize=8, color=MUTED)

    end_labels, missing = [], []
    for key, st in METHODS.items():
        res = sim["methods"].get(key)
        if res is None or "error" in res:
            if res is not None:
                missing.append(f"{st['label']}: {res['error']}")
            continue
        t = np.array(res["burn_times"]) / period
        m = np.array(res["burn_mags"])
        order = np.argsort(t)
        t, m = t[order], m[order]
        # Burn sizes: thin stems from the baseline with a ringed marker.
        ax_b.vlines(t, 0.0, m, color=st["color"], linewidth=2, zorder=2)
        ax_b.plot(t, m, st["marker"], color=st["color"], markersize=8,
                  markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3,
                  label=st["label"])
        # Cumulative delta-v: a step at each burn, held to the end.
        tt = np.concatenate([[0.0], t, [t_end]])
        cum = np.concatenate([[0.0], np.cumsum(m), [m.sum()]])
        ax_c.step(tt, cum, where="post", color=st["color"], linestyle=st["ls"],
                  linewidth=2, zorder=2, label=st["label"])
        end_labels.append((m.sum(), st["short"]))
    ax_b.set_ylim(bottom=0.0)
    ax_c.set_ylim(bottom=0.0)
    # Direct end labels with the totals, nudged apart when totals are close.
    gap = 0.07 * ax_c.get_ylim()[1]
    placed = []
    for total, short in sorted(end_labels):
        y = total if not placed else max(total, placed[-1] + gap)
        placed.append(y)
        ax_c.annotate(f"{short} {total:.3f}", (t_end * 1.02, y), xytext=(4, 0),
                      textcoords="offset points", va="center", fontsize=8,
                      color=INK2, annotation_clip=False)
    if missing:
        ax_c.text(0.98, 0.04, "\n".join(missing), transform=ax_c.transAxes,
                  ha="right", va="bottom", fontsize=8, color=MUTED)
    ax_b.set_ylabel("burn size |Δv| [m/s]", fontsize=9, color=INK2)
    ax_c.set_ylabel("cumulative Δv [m/s]", fontsize=9, color=INK2)
    ax_b.set_title("Burn sizes and timing", fontsize=10, color=INK, loc="left")
    ax_c.set_title("Cumulative fuel use (SCP: all satellites)", fontsize=10,
                   color=INK, loc="left")
    fig.legend(*ax_c.get_legend_handles_labels(), loc="outside upper center",
               ncol=len(METHODS), frameon=False, fontsize=8, labelcolor=INK2)
    plt.show()


def print_summary(sim):
    print(f"{'controller':<22}{'burns':>6}{'total Δv':>10}{'max burn':>10}"
          f"{'min sep':>9}   status")
    for key, st in METHODS.items():
        res = sim["methods"].get(key)
        if res is None or "error" in res:
            why = "not run" if res is None else res["error"]
            print(f"{st['label']:<22}{'-':>6}{'-':>10}{'-':>10}{'-':>9}   {why}")
            continue
        note = f"; {res['note']}" if res["note"] else ""
        print(f"{st['label']:<22}{len(res['burn_mags']):>6}"
              f"{res['total']:>10.4f}{max(res['burn_mags']):>10.4f}"
              f"{res['sep']:>8.0f}m   {_flags(res)}{note}")
    print("Δv in m/s. SCP totals include every satellite's burns.")


def plot_case(direction="radial", mag=0.1, delay_orb=0.25, transfer_orb=0.75,
              run_scp=True, reslot=False):
    sim = simulate(direction, mag, delay_orb, transfer_orb, run_scp, reslot)
    plot_trajectories(sim, direction, mag, delay_orb, transfer_orb)
    plot_fuel(sim, delay_orb, transfer_orb)
    print_summary(sim)

In [6]:
slider = dict(continuous_update=False, readout_format=".2f",
              layout=widgets.Layout(width="500px"),
              style={"description_width": "160px"})
widgets.interact(
    plot_case,
    direction=widgets.Dropdown(options=list(DIRECTIONS), value="radial",
                               description="hit direction",
                               style={"description_width": "160px"}),
    mag=widgets.FloatSlider(value=0.1, min=-0.5, max=0.5, step=0.01,
                            description="|δv| [m/s]", **slider),
    delay_orb=widgets.FloatSlider(value=0.25, min=0.0, max=1.5, step=0.01,
                                  description="reaction delay [orbits]",
                                  **slider),
    transfer_orb=widgets.FloatSlider(value=0.75, min=0.05, max=2.5, step=0.01,
                                     description="transfer time [orbits]",
                                     **slider),
    run_scp=widgets.Checkbox(value=True, description="run SCP (slower)",
                             style={"description_width": "160px"}),
    reslot=widgets.Checkbox(value=False, description="SCP re-slotting",
                            style={"description_width": "160px"}),
);

interactive(children=(Dropdown(description='hit direction', options=('radial', 'along-track', 'cross-track'), …